In [14]:
"""
Bradford Winter Air Quality — Single-page Dashboard (Fixed Header + Fixed Filters)

Requested changes implemented:
- Header text: "Bradford Winter Air Quality"
- Header background color: #276167
- Header full width + fixed (does not move on scroll)
- Left filter panel fixed (does not move on scroll)
- Filters panel includes a filter icon + "Reset filters" button
- Bradford logo on the header: ../assets/brad-logo-2.png
  (Dash automatically serves files placed in ./assets as /assets/<filename>)

Data input:
- ../data/processed/air_quality_hourly_clean.csv
  with columns: datetime, date, hour, week, site, pm10, pm25
"""

from __future__ import annotations

from pathlib import Path
import pandas as pd
import numpy as np

import dash
from dash import html, dcc, Input, Output, dash_table
import dash_bootstrap_components as dbc
import plotly.express as px


# =========================
# PATHS (safe for scripts + notebooks)
# =========================
# If you run this file from inside the "dashboards/" folder, BASE_DIR becomes project root.
# Adjust if your folder structure differs.
BASE_DIR = Path.cwd()
if (BASE_DIR / "data").exists() and (BASE_DIR / "assets").exists():
    # already at project root
    PROJECT_DIR = BASE_DIR
else:
    # common case: running from dashboards/ or notebooks/
    PROJECT_DIR = BASE_DIR.parent if (BASE_DIR.parent / "data").exists() else BASE_DIR

PRO_DATA = PROJECT_DIR / "data" / "processed"
OUT_DIR = PROJECT_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

HOURLY_CSV = PRO_DATA / "air_quality_hourly_clean.csv"
if not HOURLY_CSV.exists():
    raise FileNotFoundError(
        f"Missing file:\n{HOURLY_CSV}\n\n"
        "Expected location: ../data/processed/air_quality_hourly_clean.csv "
        "(relative to your project root)."
    )


# =========================
# LOAD DATA
# =========================
df = pd.read_csv(HOURLY_CSV, parse_dates=["datetime", "date"])
df["site"] = df["site"].astype(str)

# Ensure date is normalized midnight
df["date"] = pd.to_datetime(df["date"]).dt.normalize()

# Build DAILY dataset (for member-friendly trends)
daily = (
    df.groupby(["site", "date"], as_index=False)
      .agg(
          pm10_mean=("pm10", "mean"),
          pm25_mean=("pm25", "mean"),
          pm10_obs=("pm10", "count"),
          pm25_obs=("pm25", "count"),
      )
)
daily[["pm10_mean", "pm25_mean"]] = daily[["pm10_mean", "pm25_mean"]].round(2)

SITES = sorted(df["site"].unique())
DATE_MIN = daily["date"].min()
DATE_MAX = daily["date"].max()


# =========================
# DASH APP
# =========================
# Bootstrap + Bootstrap Icons for a clean filter icon
external_stylesheets = [
    dbc.themes.BOOTSTRAP,
    "https://cdn.jsdelivr.net/npm/bootstrap-icons@1.11.3/font/bootstrap-icons.css",
]
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)
app.title = "Bradford Winter Air Quality"


# =========================
# STYLING
# =========================
HEADER_BG = "#276167"
HEADER_HEIGHT_PX = 78
SIDEBAR_WIDTH_PX = 320

STYLES = {
    "header": {
        "position": "fixed",
        "top": 0,
        "left": 0,
        "right": 0,
        "height": f"{HEADER_HEIGHT_PX}px",
        "backgroundColor": HEADER_BG,
        "color": "white",
        "zIndex": 1100,
        "display": "flex",
        "alignItems": "center",
        "justifyContent": "space-between",
        "padding": "0 18px",
        "boxShadow": "0 2px 10px rgba(0,0,0,0.18)",
    },
    "header_left": {"display": "flex", "alignItems": "center", "gap": "12px"},
    "logo": {"height": "46px", "width": "auto"},
    "title_wrap": {"display": "flex", "flexDirection": "column", "gap": "2px"},
    "title": {"margin": 0, "fontSize": "22px", "fontWeight": 800, "lineHeight": "1.0"},
    "subtitle": {"margin": 0, "fontSize": "12.5px", "opacity": 0.92},

    # Fixed left sidebar
    "sidebar": {
        "position": "fixed",
        "top": f"{HEADER_HEIGHT_PX}px",
        "left": 0,
        "bottom": 0,
        "width": f"{SIDEBAR_WIDTH_PX}px",
        "backgroundColor": "#f8f9fa",
        "borderRight": "1px solid #e6e6e6",
        "padding": "14px 14px 18px 14px",
        "overflowY": "auto",
        "zIndex": 1050,
    },

    # Main content offset by fixed header + sidebar
    "content": {
        "marginTop": f"{HEADER_HEIGHT_PX}px",
        "marginLeft": f"{SIDEBAR_WIDTH_PX}px",
        "padding": "16px 18px 28px 18px",
        "backgroundColor": "white",
        "minHeight": "100vh",
    },

    "card": {
        "borderRadius": "14px",
        "boxShadow": "0 2px 10px rgba(0,0,0,0.06)",
        "border": "1px solid #efefef",
    },

    "filter_header": {
        "display": "flex",
        "alignItems": "center",
        "justifyContent": "space-between",
        "marginBottom": "10px",
    },
    "filter_title": {"margin": 0, "fontSize": "14px", "fontWeight": 800},
    "label": {"fontWeight": 700, "marginTop": "8px", "marginBottom": "6px"},
}


# =========================
# LAYOUT COMPONENTS
# =========================
header = html.Div(
    style=STYLES["header"],
    children=[
        html.Div(
            style=STYLES["header_left"],
            children=[
                # Place brad-logo-2.png inside project/assets/
                html.Img(src="/assets/brad-logo-2.png", style=STYLES["logo"]),
                html.Div(
                    style=STYLES["title_wrap"],
                    children=[
                        html.H1("Bradford Winter Air Quality", style=STYLES["title"]),
                        html.P("PM2.5 and PM10 • Winter period (Nov 2024 – Jan 2025)", style=STYLES["subtitle"]),
                    ],
                ),
            ],
        ),
    ],
)

sidebar = html.Div(
    style=STYLES["sidebar"],
    children=[
        html.Div(
            style=STYLES["filter_header"],
            children=[
                html.Div(
                    style={"display": "flex", "alignItems": "center", "gap": "8px"},
                    children=[
                        html.I(className="bi bi-funnel-fill", style={"fontSize": "16px"}),
                        html.P("Filters", style=STYLES["filter_title"]),
                    ],
                ),
                dbc.Button(
                    [html.I(className="bi bi-arrow-counterclockwise"), html.Span(" Reset", style={"marginLeft": "4px"})],
                    id="btn-reset",
                    color="secondary",
                    outline=True,
                    size="sm",
                ),
            ],
        ),

        html.Div(style=STYLES["label"], children="Site"),
        dcc.Dropdown(
            id="site",
            options=[{"label": s, "value": s} for s in SITES],
            value=SITES,
            multi=True,
            clearable=False,
            placeholder="Select site(s)...",
        ),

        html.Div(style=STYLES["label"], children="Date range"),
        dcc.DatePickerRange(
            id="date-range",
            start_date=DATE_MIN,
            end_date=DATE_MAX,
            min_date_allowed=DATE_MIN,
            max_date_allowed=DATE_MAX,
            display_format="DD/MM/YYYY",
            style={"width": "100%"},
        ),

        html.Div(style=STYLES["label"], children="Pollutant"),
        dcc.RadioItems(
            id="pollutant",
            options=[
                {"label": " PM2.5", "value": "PM2.5"},
                {"label": " PM10", "value": "PM10"},
            ],
            value="PM2.5",
            labelStyle={"display": "block", "marginBottom": "6px"},
        ),

        html.Hr(),

        dbc.Alert(
            "Tip: Hover to compare sites on the same date.",
            color="info",
            style={"fontSize": "12px", "marginBottom": "8px"},
        ),
        dbc.Alert(
            "Note: Daily means are computed from valid hourly observations.",
            color="light",
            style={"fontSize": "12px"},
        ),
    ],
)

content = html.Div(
    style=STYLES["content"],
    children=[
        dbc.Row(
            [
                dbc.Col(
                    dbc.Card(
                        dbc.CardBody(
                            [
                                html.H5("Overall daily trend", style={"marginBottom": "8px"}),
                                dcc.Graph(id="fig-overall", config={"displayModeBar": False}),
                            ]
                        ),
                        style=STYLES["card"],
                    ),
                    md=12,
                )
            ],
            className="g-3",
        ),

        dbc.Row(
            [
                dbc.Col(
                    dbc.Card(
                        dbc.CardBody(
                            [
                                html.H5("Daily trend by site", style={"marginBottom": "8px"}),
                                dcc.Graph(id="fig-by-site", config={"displayModeBar": False}),
                            ]
                        ),
                        style=STYLES["card"],
                    ),
                    md=12,
                )
            ],
            className="g-3",
            style={"marginTop": "6px"},
        ),

        dbc.Row(
            [
                dbc.Col(
                    dbc.Card(
                        dbc.CardBody(
                            [
                                html.H5("Site ranking", style={"marginBottom": "8px"}),
                                dcc.Graph(id="fig-ranking", config={"displayModeBar": False}),
                            ]
                        ),
                        style=STYLES["card"],
                    ),
                    md=12,
                )
            ],
            className="g-3",
            style={"marginTop": "6px"},
        ),

        dbc.Row(
            [
                dbc.Col(
                    dbc.Card(
                        dbc.CardBody(
                            [
                                html.H5("Daily summary table", style={"marginBottom": "8px"}),
                                dash_table.DataTable(
                                    id="tbl-daily",
                                    page_size=12,
                                    sort_action="native",
                                    filter_action="native",
                                    style_table={"overflowX": "auto"},
                                    style_header={"fontWeight": "800"},
                                    style_cell={
                                        "fontFamily": "Arial",
                                        "fontSize": "12px",
                                        "padding": "6px",
                                        "whiteSpace": "nowrap",
                                    },
                                ),
                            ]
                        ),
                        style=STYLES["card"],
                    ),
                    md=12,
                )
            ],
            className="g-3",
            style={"marginTop": "6px"},
        ),
    ],
)

app.layout = html.Div([header, sidebar, content])


# =========================
# HELPERS
# =========================
def filter_daily(selected_sites, start_date, end_date):
    d = daily.copy()
    if selected_sites:
        d = d[d["site"].isin(selected_sites)]
    if start_date is not None:
        d = d[d["date"] >= pd.to_datetime(start_date).normalize()]
    if end_date is not None:
        d = d[d["date"] <= pd.to_datetime(end_date).normalize()]
    return d


def polish(fig, title, y_title):
    fig.update_layout(
        title=title,
        hovermode="x unified",
        template="plotly_white",
        margin=dict(l=40, r=20, t=60, b=40),
        height=420,
        legend_title_text="",
    )
    fig.update_xaxes(title_text="Date", tickfont=dict(size=12), title_font=dict(size=13))
    fig.update_yaxes(title_text=y_title, tickfont=dict(size=12), title_font=dict(size=13))
    return fig


# =========================
# CALLBACKS
# =========================
@app.callback(
    Output("site", "value"),
    Output("date-range", "start_date"),
    Output("date-range", "end_date"),
    Output("pollutant", "value"),
    Input("btn-reset", "n_clicks"),
    prevent_initial_call=True,
)
def reset_filters(_):
    return SITES, DATE_MIN, DATE_MAX, "PM2.5"


@app.callback(
    Output("fig-overall", "figure"),
    Output("fig-by-site", "figure"),
    Output("fig-ranking", "figure"),
    Output("tbl-daily", "data"),
    Output("tbl-daily", "columns"),
    Input("site", "value"),
    Input("date-range", "start_date"),
    Input("date-range", "end_date"),
    Input("pollutant", "value"),
)
def update(selected_sites, start_date, end_date, pollutant):
    d = filter_daily(selected_sites, start_date, end_date)

    if pollutant == "PM2.5":
        value_col = "pm25_mean"
        y_title = "PM2.5 (µg/m³)"
        overall_title = "Overall daily PM2.5 (mean across selected sites)"
        by_site_title = "Daily PM2.5 by site"
        rank_title = "Site ranking by average daily PM2.5"
    else:
        value_col = "pm10_mean"
        y_title = "PM10 (µg/m³)"
        overall_title = "Overall daily PM10 (mean across selected sites)"
        by_site_title = "Daily PM10 by site"
        rank_title = "Site ranking by average daily PM10"

    # Overall trend across selected sites
    overall = (
        d.groupby("date", as_index=False)
         .agg(value=(value_col, "mean"))
    )
    fig_overall = px.line(overall, x="date", y="value")
    fig_overall = polish(fig_overall, overall_title, y_title)

    # By-site trend
    fig_by_site = px.line(d, x="date", y=value_col, color="site")
    fig_by_site = polish(fig_by_site, by_site_title, y_title)

    # Ranking (average daily within selected filters)
    ranking = (
        d.groupby("site", as_index=False)
         .agg(
             avg_daily=(value_col, "mean"),
             days_with_data=(value_col, lambda s: int(s.notna().sum())),
         )
    )
    ranking["avg_daily"] = ranking["avg_daily"].round(2)
    ranking = ranking.sort_values("avg_daily", ascending=False)

    fig_rank = px.bar(
        ranking,
        x="avg_daily",
        y="site",
        orientation="h",
        hover_data={"days_with_data": True},
        labels={"avg_daily": f"Average daily {pollutant} (µg/m³)", "site": "Site"},
        title=rank_title,
    )
    fig_rank.update_layout(template="plotly_white", height=420, margin=dict(l=40, r=20, t=60, b=40))
    fig_rank.update_yaxes(categoryorder="total ascending", tickfont=dict(size=13), title_font=dict(size=13))
    fig_rank.update_xaxes(tickfont=dict(size=13), title_font=dict(size=13))

    # Table
    table_cols = ["site", "date", "pm10_mean", "pm25_mean", "pm10_obs", "pm25_obs"]
    table = d[table_cols].copy()
    table["date"] = table["date"].dt.strftime("%d/%m/%Y")

    columns = [
        {"name": "Site", "id": "site"},
        {"name": "Date", "id": "date"},
        {"name": "PM10 daily mean", "id": "pm10_mean"},
        {"name": "PM2.5 daily mean", "id": "pm25_mean"},
        {"name": "PM10 observed (hours)", "id": "pm10_obs"},
        {"name": "PM2.5 observed (hours)", "id": "pm25_obs"},
    ]

    return fig_overall, fig_by_site, fig_rank, table.to_dict("records"), columns


# =========================
# RUN
# =========================
if __name__ == "__main__":
    app.run(debug=True, host="127.0.0.1", port=8051)

[2026-01-11 23:12:07,187] ERROR in app: Exception on /assets/brad-logo-2.png [GET]
Traceback (most recent call last):
  File "/Users/sayo2rule/.local/pipx/venvs/jupyter/lib/python3.14/site-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  File "/Users/sayo2rule/.local/pipx/venvs/jupyter/lib/python3.14/site-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "/Users/sayo2rule/.local/pipx/venvs/jupyter/lib/python3.14/site-packages/flask/blueprints.py", line 100, in send_static_file
    return send_from_directory(
        t.cast(str, self.static_folder), filename, max_age=max_age
    )
  File "/Users/sayo2rule/.local/pipx/venvs/jupyter/lib/python3.14/site-packages/flask/helpers.py", line 572, in send_from_directory
    return werkzeug.utils.send_from_dire